# Lab 09: State Estimation and the Kalman Filter
**Computational Sensorimotor Control — Week 9 | **

This lab builds the SPFC pipeline from the lecture notes (§2–§6).
Each part implements one block of the State-Predicting Feedback Controller
(Frens & Donchin, 2009).

| Part | Lecture | What you build | Source |
|------|---------|----------------|--------|
| 1 | §1 | Proprioceptive sensor | van Beers et al., 1998 |
| 2 | §1 | Dark controller (P-only with proprio) | — |
| 3 | §3 | Forward model (predict step) | Frens & Donchin, Eq. 2–3 |
| 4a | §4 | Scalar Bayesian fusion (static) | van Beers et al., 1999; Ernst & Banks, 2002 |
| 4b | §4 | Fused controller (multi-delay) | Crevecoeur et al., 2016, Eq. 8 |
| 5 | §5–§6 | Arc comparison + frontier | — |


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'})

from smc import Arm, Q_REF, mass_matrix, coriolis, joint_accelerations, Sensor

arm = Arm()
fk  = arm.forward_kinematics
def ik(p): return arm.inverse_kinematics(p[0], p[1])
Jfn = arm.jacobian
dt  = 0.001

start_hand = np.array(fk(Q_REF))
Kp = 8; k_noise = 0.15

NAVY='#1B2A4A'; TEAL='#2E86AB'; RED='#E74C3C'
GREEN='#27AE60'; GRAY='#7F8C8D'; ORANGE='#E67E22'; PURPLE='#8E44AD'

# ── Reusable helpers (from lecture figure scripts) ──
def add_sdn(tau, k, rng):
    return tau + rng.normal(0, k * np.abs(tau))

def minjerk_reach(tgt, T):
    n = int(T/dt) + 1; t = np.linspace(0, T, n)
    s = 10*(t/T)**3 - 15*(t/T)**4 + 6*(t/T)**5
    return t, start_hand[None,:] + s[:,None] * (tgt - start_hand)[None,:]

def minjerk_arc(R, T):
    n = int(T/dt) + 1; t = np.linspace(0, T, n)
    s = 10*(t/T)**3 - 15*(t/T)**4 + 6*(t/T)**5
    theta = np.pi * s; cx, cy = start_hand[0], start_hand[1] + R
    return t, np.column_stack([cx + R*np.sin(theta), cy - R*np.cos(theta)])

def id_torques(t, pos):
    q = np.array([ik(p) for p in pos])
    qd = np.gradient(q, dt, axis=0); qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros_like(q)
    for i in range(len(q)):
        tau[i] = mass_matrix(q[i]) @ qdd[i] + coriolis(q[i], qd[i])
    return q, qd, qdd, tau


---
## Part 1: The Proprioceptive Sensor (Lecture §1)

Proprioception provides a noisy, delayed measurement of hand position
(van Beers et al., 1998; Lecture §1, BOX 2):

$$y_{pr}(t) = x_{hand}(t - \Delta_{pr}) + \varepsilon(t), \quad \varepsilon \sim \mathcal{N}(0, R_{pr})$$

where:
- $y_{pr}(t)$ (2×1): proprioceptive measurement of hand position
- $\Delta_{pr} = 60$ ms: cortical proprioceptive delay
- $R_{pr} = \sigma_{pr}^2 I_2$ (2×2): measurement noise covariance (isotropic approximation)
- $\sigma_{pr} = 12$ mm: hand-position noise (from the Supplementary noise chain)

**Task:** Fill in the delay (seconds) and noise covariance matrix $R_{pr}$ (2×2).


In [ ]:
# ══ FILL IN: proprioceptive sensor parameters ══
DELAY_PR = ...  # ← YOUR CODE HERE
R_PR     = ...  # ← YOUR CODE HERE

proprio = Sensor(delay=DELAY_PR, R=R_PR)
SIGMA_PR = np.sqrt(R_PR[0,0])
print(f'Delay: {DELAY_PR*1000:.0f} ms, σ: {SIGMA_PR*1000:.0f} mm')


In [ ]:
# Verify on a 10 cm reach: sample every 10 ms (matching Lecture Figure 1D)
tgt = start_hand + np.array([0.10, 0]); T = 0.6
t_arr, pos_des = minjerk_reach(tgt, T)
q_traj, qd_traj, _, tau_ff = id_torques(t_arr, pos_des)
hand_true = np.array([fk(q_traj[i]) for i in range(len(t_arr))])

# Sample at 10 ms intervals with 60 ms delay
sample_dt = 0.010  # 10 ms
sample_steps = int(sample_dt / dt)
delay_steps = int(DELAY_PR / dt)
proprio.reset(); rng = np.random.default_rng(42)

sample_times = []; sample_meas = []; sample_true = []
for i in range(0, len(t_arr), sample_steps):
    idx_delayed = max(0, i - delay_steps)
    y = pos_des[idx_delayed] + rng.normal(0, SIGMA_PR, 2)
    sample_times.append(t_arr[i])
    sample_meas.append(y)
    sample_true.append(pos_des[i])  # true at measurement time (not delayed)
sample_times = np.array(sample_times) * 1000  # ms
sample_meas = np.array(sample_meas)
sample_true = np.array(sample_true)

t_ms = t_arr * 1000  # used throughout the lab

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel A: true trajectory + discrete proprio samples
ax1.plot(t_arr*1000, pos_des[:,0]*100, color=GRAY, lw=2.5, label='True x')
ax1.scatter(sample_times, sample_meas[:,0]*100, c=PURPLE, s=20, marker='s',
            zorder=5, label=f'Proprio samples (Δ={DELAY_PR*1000:.0f}ms, every 10ms)')
ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('x (cm)')
ax1.set_title('A: Discrete proprioceptive samples', fontweight='bold'); ax1.legend(fontsize=8)

# Panel B: error at each sample point
valid = sample_times >= DELAY_PR * 1000
err_mm = np.linalg.norm(sample_meas[valid] - sample_true[valid], axis=1) * 1000
ax2.scatter(sample_times[valid], err_mm, c=PURPLE, s=15, zorder=5)
ax2.axhline(SIGMA_PR*1000, color='k', ls='--', label=f'σ = {SIGMA_PR*1000:.0f} mm')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Error (mm)')
ax2.set_title('B: Measurement error at each sample', fontweight='bold'); ax2.legend()
plt.suptitle('Part 1: Proprioceptive sensor — discrete samples at 10 ms intervals', fontweight='bold')
plt.tight_layout(); plt.show()


---
## Part 2: The Dark Controller (Lecture §1)

A P-only feedback controller using proprioception (Lecture BOX 2):

$$e(t) = x_{des}(t - \Delta_{pr}) - y_{pr}(t) \quad (2 \times 1 \text{ position error})$$

$$\tau_{corr} = J(q)^T \cdot K_p \cdot e \quad (2 \times 2)(\text{scalar})(2 \times 1) \rightarrow (2 \times 1 \text{ joint torques})$$

$$\tau = \tau_{ff} + \tau_{corr} + \text{SDN} \quad (2 \times 1)$$

where $K_p = 8$ and SDN is signal-dependent noise ($k = 0.15$).

**Task:** Inside `sim_p`, fill in the 2 lines that compute the noisy measurement
and the correction torque.


In [ ]:
def sim_p(q0, tau_ff, pos_des, Kp, delay_s, sigma, rng):
    """Single-sensor P controller (from lecture Figure 2 script).
    Uses a single RNG for motor + sensor noise — matches the lecture."""
    n = len(tau_ff)
    qs = np.zeros((n,2)); qds = np.zeros((n,2)); hs = np.zeros((n,2))
    qs[0] = q0.copy(); hs[0] = fk(q0)
    buf = []; bl = int(delay_s / dt)
    for i in range(n-1):
        buf.append(hs[i].copy()); tc = np.zeros(2)
        if Kp > 0 and len(buf) > bl:
            # ══ FILL IN (2 lines): noisy measurement y, then correction torque tc ══
            y  = ...  # ← YOUR CODE HERE
            tc = ...  # ← YOUR CODE HERE
        tt = add_sdn(tau_ff[i] + tc, k_noise, rng)
        qdd_i = joint_accelerations(qs[i], qds[i], tt)
        qds[i+1] = qds[i] + qdd_i*dt
        qs[i+1]  = qs[i]  + qds[i+1]*dt
        hs[i+1]  = fk(qs[i+1])
    return hs


In [ ]:
# ── Reproduce Figure 2: dark accuracy on the 6 cm arc ──
R_ARC = 0.06; T_ARC = 0.8; N = 200
t_a, pos_a = minjerk_arc(R_ARC, T_ARC)
q_a, _, _, tau_a = id_torques(t_a, pos_a)
tgt_a = pos_a[-1]

configs = [
    ('Open-loop',   0,  0,         0,         RED),
    ('Dark',        Kp, DELAY_PR,  SIGMA_PR,  PURPLE),
    ('Peripheral',  Kp, 0.080,     0.005,     ORANGE),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5)); sds = []
for name, kp, dly, sig, col in configs:
    eps = np.array([sim_p(q_a[0],tau_a,pos_a,kp,dly,sig,
                          np.random.default_rng(i))[-1]
                    for i in range(N)])
    sds.append(np.std(np.linalg.norm(eps - tgt_a, axis=1)) * 100)
    for j in range(25):
        h = sim_p(q_a[0],tau_a,pos_a,kp,dly,sig,np.random.default_rng(200+j))
        axes[0].plot(h[:,0]*100, h[:,1]*100, color=col, lw=0.5, alpha=0.3)
    axes[1].scatter(eps[:,0]*100, eps[:,1]*100, c=col, s=5, alpha=0.3, label=name)

axes[0].set_aspect('equal'); axes[0].set_title('A: Hand paths', fontweight='bold')
axes[1].set_aspect('equal'); axes[1].legend(fontsize=8)
axes[1].set_title('B: Endpoint scatter', fontweight='bold')
bars = axes[2].bar(range(3), sds,
                   color=[c for _,_,_,_,c in configs], edgecolor='k', lw=0.5)
for b,s in zip(bars,sds):
    axes[2].text(b.get_x()+b.get_width()/2, s+0.003, f'{s:.3f}', ha='center', fontsize=9)
axes[2].set_xticks(range(3))
axes[2].set_xticklabels(['Open\nloop','Dark\n(proprio)','Peripheral'])
axes[2].set_ylabel('Endpoint SD (cm)')
axes[2].set_title('C: Endpoint SD', fontweight='bold')
plt.suptitle('Figure 2: Dark accuracy (6 cm arc, T=800 ms, Kp=8)', fontweight='bold')
plt.tight_layout(); plt.show()
for (n,_,_,_,_), s in zip(configs, sds): print(f'  {n}: {s:.3f} cm')


---
## Part 3: The Forward Model — Predict Step (Lecture §3; Frens & Donchin, 2009, Eqs. 2–3)

The arm's true dynamics (Frens & Donchin, Eq. 2):

$$x_{n+1} = A\, x_n + B\, u_n + w_n$$

The forward model prediction (Frens & Donchin, Eq. 3):

$$\hat{x}_{FM,n+1} = A\, \hat{x}_n + B\, u_n \quad \text{(efference copy — no noise term)}$$

where:
- $x = [x, y, v_x, v_y]^T$ (4×1): state vector
- $u_n = [\tau_{sh}, \tau_{el}]^T$ (2×1): motor command (efference copy)
- $A$ (4×4): state transition — propagates position using velocity
- $B$ (4×2): maps torques → hand acceleration via $J(q) M(q)^{-1}$
- $w_n \sim \mathcal{N}(0, Q)$ (4×1): process noise (SDN) — **unknown** to the forward model

Here we use a simplified position-only forward model: advance the predicted
hand position using the desired velocity from the plan.

**Task:** Fill in the forward model update — advance the prediction by one timestep (1 line).


In [ ]:
vel_des = np.gradient(pos_des, dt, axis=0)

fwd_pred = np.zeros_like(hand_true)
fwd_pred[0] = hand_true[0].copy()
for i in range(len(t_arr) - 1):
    # ══ FILL IN: advance prediction using desired velocity (1 line) ══
    fwd_pred[i+1] = ...  # ← YOUR CODE HERE

err_mm = np.linalg.norm(fwd_pred - hand_true, axis=1) * 1000
print(f'Forward model error at end: {err_mm[-1]:.1f} mm (should be ~0 without SDN)')


### Target jump: the forward model doesn't know

At $t = 250$ ms, the target jumps 3 cm upward. The true hand steers toward the
new target, but the forward model continues on the **original plan** because
efference copy reflects the plan, not the perturbation.

The predict + update estimate gradually corrects as delayed sensory measurements
arrive. This demonstrates why the update step is needed.


In [ ]:
# ── Figure 3: Forward model with target jump (from lecture script) ──
tgt_new = tgt + np.array([0.0, 0.03])  # 3 cm upward
jump_step = int(0.250 / dt)

# Actual trajectory: re-plan from jump point to new target
T_remain = T - 0.250
pos_at_jump = pos_des[jump_step]
t_remain = t_arr[jump_step:] - t_arr[jump_step]
s2 = 10*(t_remain/T_remain)**3 - 15*(t_remain/T_remain)**4 + 6*(t_remain/T_remain)**5
pos_actual = np.copy(pos_des)
pos_actual[jump_step:] = pos_at_jump[None,:] + s2[:,None]*(tgt_new - pos_at_jump)[None,:]
vel_actual = np.gradient(pos_actual, dt, axis=0)

# Simulate true hand with SDN on the actual trajectory
q_act = np.array([ik(p) for p in pos_actual])
qd_act = np.gradient(q_act, dt, axis=0); qdd_act = np.gradient(qd_act, dt, axis=0)
tau_act = np.zeros_like(q_act)
for i in range(len(q_act)):
    tau_act[i] = mass_matrix(q_act[i])@qdd_act[i] + coriolis(q_act[i],qd_act[i])

rng_ex = np.random.default_rng(5)
qs_ex = np.zeros_like(q_act); qds_ex = np.zeros_like(q_act); hs_ex = np.zeros((len(t_arr),2))
qs_ex[0] = q_act[0]; hs_ex[0] = fk(q_act[0])
for i in range(len(t_arr)-1):
    tt = add_sdn(tau_act[i], k_noise, rng_ex)
    qdd_i = joint_accelerations(qs_ex[i], qds_ex[i], tt)
    qds_ex[i+1] = qds_ex[i]+qdd_i*dt; qs_ex[i+1] = qs_ex[i]+qds_ex[i+1]*dt
    hs_ex[i+1] = fk(qs_ex[i+1])

# Forward model: uses ORIGINAL plan velocity (doesn't know about jump)
fwd_ex = np.zeros_like(hs_ex); fwd_ex[0] = hs_ex[0].copy()
for i in range(len(t_arr)-1):
    fwd_ex[i+1] = fwd_ex[i] + vel_des[i] * dt

# Predict + update: blends forward model with delayed peripheral measurements
K_blend = 0.3; bl_pv = int(0.080/dt); ss = int(0.010/dt)
rng_m = np.random.default_rng(42)
fwd_upd = np.zeros_like(hs_ex); fwd_upd[0] = hs_ex[0].copy()
for i in range(len(t_arr)-1):
    pred = fwd_upd[i] + vel_des[i] * dt
    if (i+1) >= bl_pv and (i+1) % ss == 0:
        idx_d = max(0, (i+1) - bl_pv)
        y_meas = hs_ex[idx_d] + rng_m.normal(0, 0.005, 2)
        # Project measurement forward to current time using plan velocity
        proj = y_meas.copy()
        for k in range(idx_d, min(i+1, len(t_arr)-1)): proj += vel_des[k]*dt
        pred = pred + K_blend * (proj - pred)
    fwd_upd[i+1] = pred

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t_ms, hs_ex[:,1]*100, color=GRAY, lw=2.5, label='True hand')
ax1.plot(t_ms, fwd_ex[:,1]*100, color=TEAL, lw=2, ls='--', label='Predict only')
ax1.plot(t_ms, fwd_upd[:,1]*100, color=PURPLE, lw=2, label='Predict + update')
ax1.axvline(250, color=RED, ls=':', lw=1.5)
ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('y (cm)'); ax1.legend(fontsize=8)
ax1.set_title('A: y-position with target jump at 250 ms', fontweight='bold')

ax2.plot(t_ms, np.linalg.norm(fwd_ex-hs_ex, axis=1)*1000, color=TEAL, lw=2, ls='--', label='Predict only')
ax2.plot(t_ms, np.linalg.norm(fwd_upd-hs_ex, axis=1)*1000, color=PURPLE, lw=2, label='Predict + update')
ax2.axvline(250, color=RED, ls=':', lw=1.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Error (mm)'); ax2.legend()
ax2.set_title('B: Prediction error', fontweight='bold')
plt.suptitle('Figure 3: Forward model drift — target jump at 250 ms', fontweight='bold')
plt.tight_layout(); plt.show()


---
## Part 4a: Scalar Bayesian Fusion — Static Case
**(Lecture §4; van Beers et al., 1999; Ernst & Banks, 2002)**

When both sensors report the **same moment in time** (static case):

$$\hat{x}_{fused} = \frac{y_1 / \sigma_1^2 + y_2 / \sigma_2^2}{1/\sigma_1^2 + 1/\sigma_2^2}$$

$$\sigma^2_{fused} = \frac{1}{1/\sigma_1^2 + 1/\sigma_2^2}$$

With $\sigma_{periph} = 5$ mm and $\sigma_{proprio} = 12$ mm:

**Task:** Compute $\sigma_{fused}$ (1 line).


In [ ]:
sigma_periph = 5.0   # mm
sigma_proprio = 12.0  # mm

# ══ FILL IN: compute sigma_fused (1 line) ══
sigma_fused = ...  # ← YOUR CODE HERE

print(f'σ_periph  = {sigma_periph:.0f} mm')
print(f'σ_proprio = {sigma_proprio:.0f} mm')
print(f'σ_fused   = {sigma_fused:.1f} mm  (< min of both: {min(sigma_periph,sigma_proprio):.0f} mm)')
print(f'Vision weight: {(1/sigma_periph**2)/(1/sigma_periph**2+1/sigma_proprio**2)*100:.0f}%')


In [ ]:
# ── Figure 5: Bayesian fusion (from lecture script) ──
mu_pv = 2.0; mu_pr = -1.5
w_pv = 1/sigma_periph**2; w_pr = 1/sigma_proprio**2
mu_fused = (w_pv*mu_pv + w_pr*mu_pr) / (w_pv + w_pr)

x = np.linspace(-30, 30, 500)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, norm.pdf(x, mu_pv, sigma_periph), color=ORANGE, lw=2,
        label=f'Peripheral (σ={sigma_periph:.0f} mm)')
ax.plot(x, norm.pdf(x, mu_pr, sigma_proprio), color=PURPLE, lw=2,
        label=f'Proprioception (σ={sigma_proprio:.0f} mm)')
ax.plot(x, norm.pdf(x, mu_fused, sigma_fused), color=TEAL, lw=3,
        label=f'Fused (σ={sigma_fused:.1f} mm)')
ax.fill_between(x, norm.pdf(x, mu_fused, sigma_fused), alpha=0.15, color=TEAL)
ax.set_xlabel('Position error (mm)'); ax.set_ylabel('Probability density')
ax.set_title('Figure 5: Static fusion — valid when both sensors report the same moment',
             fontweight='bold')
ax.legend(loc='upper left'); plt.tight_layout(); plt.show()


---
## Part 4b: Multi-Sensor Fused Controller
**(Lecture §4–§5; Crevecoeur et al., 2016, Eq. 8; Frens & Donchin, 2009, Eq. 5)**

The Kalman update (Frens & Donchin, Eq. 5):

$$\hat{x}_{n+1} = \hat{x}_{FM,n+1} + K_n (y_n - H \hat{x}_{FM,n+1})$$

With multi-delay fusion (Crevecoeur et al., Eq. 8):

$$\hat{x}(t) = \hat{x}_{FM}(t) + K_P(y_{proprio} - \hat{x}(t-\Delta_p)) + K_V(y_{vision} - \hat{x}(t-\Delta_v))$$

Each sensor's prediction error is computed at **its own delay time**.
We align measurements to the newest sensor (proprioception, $\Delta_p$ = 60 ms)
by projecting older measurements forward using the desired velocity.

The sensor errors are combined using **inverse-variance weighting**:

$$e_{fused} = \frac{\sum_j w_j \cdot e_j}{\sum_j w_j}, \quad w_j = \frac{1}{\sigma_j^2}$$

**Task:** Fill in the 2 lines that compute the weighted error and correction torque.


In [ ]:
PROPRIO = ...  # ← YOUR CODE HERE


---
## Part 5: Fused Controller on the Arc and the Frontier (Lecture §5–§6)

The three-sensor controller (Lecture BOX 11):

$$\hat{x}_{hand} = [\hat{x}, \hat{y}, \hat{v}_x, \hat{v}_y]^T \quad (4 \times 1)$$
$$\hat{x}_{target} = [\hat{x}_{tgt}, \hat{y}_{tgt}]^T \quad (2 \times 1)$$
$$e(t) = \hat{x}_{target} - H_{pos}\, \hat{x}_{hand} \quad (2 \times 1 - (2 \times 4)(4 \times 1) \rightarrow 2 \times 1)$$

where $H_{pos} = \begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}$ (2×4) extracts position.

During movement, proprioception dominates the estimate because its 60 ms delay
provides **newer** information than vision's 80 ms (Crevecoeur et al., 2016).
The fusion benefit is modest at fast speeds and grows at slow speeds.


In [ ]:
# ── Figure 8: Fused controller on the 6 cm arc ──
# sim_p and sim_fused_p both use single RNG — fair comparison
configs_arc = [
    ('Open-loop',      lambda s: sim_p(q_a[0],tau_a,pos_a,0,0,0,np.random.default_rng(s)),                      RED),
    ('Peripheral',     lambda s: sim_p(q_a[0],tau_a,pos_a,Kp,0.080,0.005,np.random.default_rng(s)),             ORANGE),
    ('Proprio (dark)', lambda s: sim_p(q_a[0],tau_a,pos_a,Kp,DELAY_PR,SIGMA_PR,np.random.default_rng(s)),       PURPLE),
    ('Fused',          lambda s: sim_fused_p(q_a[0],tau_a,pos_a,Kp,np.random.default_rng(s),[PROPRIO,PERIPH]),GREEN),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5)); sds_arc = []
for name, fn, col in configs_arc:
    eps = np.array([fn(i)[-1] for i in range(N)])
    sds_arc.append(np.std(np.linalg.norm(eps - tgt_a, axis=1)) * 100)
    for j in range(25):
        h = fn(200+j)
        axes[0].plot(h[:,0]*100, h[:,1]*100, color=col, lw=0.5, alpha=0.3)
    axes[1].scatter(eps[:,0]*100, eps[:,1]*100, c=col, s=5, alpha=0.3, label=name)

axes[0].set_aspect('equal'); axes[0].set_title('A: Hand paths', fontweight='bold')
axes[1].set_aspect('equal'); axes[1].legend(fontsize=8)
axes[1].set_title('B: Endpoint scatter', fontweight='bold')
names_bar = ['Open\nloop','Periph.','Proprio.\n(dark)','Fused']
bars = axes[2].bar(range(4), sds_arc,
                   color=[c for _,_,c in configs_arc], edgecolor='k', lw=0.5)
for b,s in zip(bars,sds_arc):
    axes[2].text(b.get_x()+b.get_width()/2, s+0.003, f'{s:.3f}', ha='center', fontsize=9)
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(names_bar, fontsize=9)
axes[2].set_ylabel('Endpoint SD (cm)')
axes[2].set_title('C: Endpoint SD', fontweight='bold')
plt.suptitle('Figure 8: Fused controller on 6 cm arc (T=800 ms, Kp=8)', fontweight='bold')
plt.tight_layout(); plt.show()
for (n,_,_),s in zip(configs_arc,sds_arc): print(f'  {n}: {s:.3f} cm')


### The Frontier (Figure 9)

Sweep movement times from 250 ms to 1000 ms with all 5 controllers.

**Note:** Proprioception alone may degrade at slow speeds — this is a
limitation of the fixed-gain P controller ($K_p = 8$), not of proprioception
itself. A proper Kalman gain would automatically reduce corrections as
uncertainty stabilizes near the endpoint (Lecture §6 caption).


In [ ]:
# ── Figure 9: The updated frontier ──
# sim_fused_frontier uses separated RNGs for fair cross-speed comparison
T_vals = [0.25, 0.30, 0.40, 0.50, 0.60, 0.80, 1.0]
N_mc = 150
tgt_r = start_hand + np.array([0.10, 0])
sds_all = {k: [] for k in ['ol','pv','cv','pr','fu']}

for Ti in T_vals:
    t_r, pos_r = minjerk_reach(tgt_r, Ti)
    q_r, _, _, tau_r = id_torques(t_r, pos_r)
    for key, dly, sig in [('ol',0,0),('pv',0.080,0.005),('cv',0.150,0.001),
                           ('pr',DELAY_PR,SIGMA_PR)]:
        kp_use = 0 if key=='ol' else Kp
        eps = np.array([sim_p(q_r[0],tau_r,pos_r,kp_use,dly,sig,
                              np.random.default_rng(i))[-1]
                        for i in range(N_mc)])
        sds_all[key].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    # Fused: separated RNGs
    eps = np.array([sim_fused_frontier(q_r[0],tau_r,pos_r,Kp,[PROPRIO,PERIPH],i)[-1]
                    for i in range(N_mc)])
    sds_all['fu'].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    print(f'  T={Ti*1000:.0f} ms done')

T_ms = [t*1000 for t in T_vals]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(T_ms,sds_all['ol'],'o--',color=RED,lw=2,ms=6,label='Open-loop')
ax.plot(T_ms,sds_all['cv'],'s-',color=TEAL,lw=1.5,ms=5,label=r'Central ($\Delta$=150ms, $\sigma$=1mm)')
ax.plot(T_ms,sds_all['pv'],'^-',color=ORANGE,lw=1.5,ms=5,label=r'Peripheral ($\Delta$=80ms, $\sigma$=5mm)')
ax.plot(T_ms,sds_all['pr'],'D-',color=PURPLE,lw=1.5,ms=5,label=r'Proprio ($\Delta$=60ms, $\sigma$=12mm)')
ax.plot(T_ms,sds_all['fu'],'p-',color=GREEN,lw=2.5,ms=7,label='Fused (proprio+periph)',zorder=5)
ax.fill_between(T_ms, sds_all['fu'], 0, alpha=0.08, color=GREEN)
ax.set_xlabel('Movement time (ms)'); ax.set_ylabel('Endpoint SD (cm)')
ax.set_title('Figure 9: The updated frontier', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(200, 1050)
plt.tight_layout(); plt.show()


---
## Summary

You have built the complete SPFC pipeline:

1. **Proprioceptive sensor** — $y_{pr} = x(t-60\text{ms}) + \varepsilon$, $\sigma = 12$ mm
2. **Dark controller** — P-only with proprio beats open-loop by ~38%
3. **Forward model** — predicts through delay using efference copy; drifts without updates
4. **Bayesian fusion** — static: σ_fused = 4.6 mm (vision dominates 85%); dynamic: delays shift weight toward proprioception (Crevecoeur et al., 2016)
5. **Fused frontier** — fusion helps modestly at fast speeds, more at slow speeds

**Key insight (Crevecoeur et al., 2016):** During movement, somatosensory *speed*
trumps visual *accuracy*. The 85%/15% vision/proprio split applies only when the
hand is stationary.
